<a href="https://colab.research.google.com/github/EmirhanByr/stock-market-elt-pipeline/blob/main/01_ingest_market_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q yfinance google-cloud-bigquery pyarrow

import yfinance as yf
import pandas as pd
from datetime import datetime

# 1. PARAMETRELER VE KAYNAKLAR
HISSELER = ["THYAO.IS", "ASELS.IS", "TUPRS.IS", "GARAN.IS", "EREGL.IS"]
DOVIZ_KODU = "USDTRY=X"  # Yahoo Finance Dolar/TL sembolü
ZAMAN_ARALIGI = "3mo"     # Daha zengin zaman serisi ve AI eğitimi için son 3 ay

# 2. SEKTÖR BOYUT TABLOSU (dim_sirketler)
# İleride SQL'de JOIN atacağımız referans verisi
sirket_bilgileri = {
    "hisse_kodu": ["THYAO", "ASELS", "TUPRS", "GARAN", "EREGL"],
    "sirket_adi": ["Türk Hava Yolları", "Aselsan", "Tüpraş", "Garanti BBVA", "Ereğli Demir Çelik"],
    "sektor": ["Havacılık", "Savunma Sanayi", "Enerji / Petrol", "Bankacılık", "Demir Çelik"]
}
df_dim_sirketler = pd.DataFrame(sirket_bilgileri)

# 3. DOLAR KURU VERİSİNİ ÇEKME (raw_doviz_kurlari)
print("⏳ Dolar/TL kuru çekiliyor...")
dolar_raw = yf.Ticker(DOVIZ_KODU).history(period=ZAMAN_ARALIGI).reset_index()
df_dolar = pd.DataFrame({
    "islem_tarihi": pd.to_datetime(dolar_raw["Date"]).dt.strftime("%Y-%m-%d"),
    "usd_try_kapanis": dolar_raw["Close"].round(4)
})

# 4. BIST HİSSE VERİLERİNİ ÇEKME (raw_hisse_fiyatlari)
print("⏳ BIST hisse verileri çekiliyor...")
hisse_listesi = []
for sembol in HISSELER:
    raw = yf.Ticker(sembol).history(period=ZAMAN_ARALIGI).reset_index()
    raw["hisse_kodu"] = sembol.replace(".IS", "")
    hisse_listesi.append(raw)

ham_hisseler = pd.concat(hisse_listesi, ignore_index=True)

df_hisseler = pd.DataFrame({
    "islem_tarihi": pd.to_datetime(ham_hisseler["Date"]).dt.strftime("%Y-%m-%d"),
    "hisse_kodu": ham_hisseler["hisse_kodu"],
    "acilis_fiyati": ham_hisseler["Open"].round(2),
    "kapanis_fiyati": ham_hisseler["Close"].round(2),
    "en_yuksek": ham_hisseler["High"].round(2),
    "en_dusuk": ham_hisseler["Low"].round(2),
    "islem_hacmi": ham_hisseler["Volume"].astype("int64"),
    "yuklenme_tarihi": datetime.now().strftime("%Y-%m-%d")
})

# Kontrol çıktıları
print(f"✅ Şirket Boyut Tablosu: {len(df_dim_sirketler)} kayıt")
print(f"✅ Dolar Kuru Tablosu: {len(df_dolar)} gün")
print(f"✅ Hisse Fiyatları Tablosu: {len(df_hisseler)} satır (5 hisse x 3 ay)")

⏳ Dolar/TL kuru çekiliyor...
⏳ BIST hisse verileri çekiliyor...
✅ Şirket Boyut Tablosu: 5 kayıt
✅ Dolar Kuru Tablosu: 66 gün
✅ Hisse Fiyatları Tablosu: 330 satır (5 hisse x 3 ay)


In [2]:
from google.colab import auth
from google.cloud import bigquery

# 1. Google Cloud Kimlik Doğrulama
# Çalıştırdığında bir onay penceresi açılacak, Google hesabına izin ver
auth.authenticate_user()

# 2. BigQuery Bağlantısı
PROJECT_ID = "veri-muhendisligi-101-511413"
DATASET_ID = "borsa_analizi"
client = bigquery.Client(project=PROJECT_ID)

# Mühendislik Stratejisi: WRITE_TRUNCATE
# Boru hattını her gün yeniden çalıştırdığında veriyi üst üste yazıp (duplicate) etmez,
# her zaman en güncel halini temiz bir şekilde üzerine yazar (Idempotent Pipeline).
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

# 3. Üç Tabloyu Sırayla Yükleme
tablolar = {
    "dim_sirketler": df_dim_sirketler,
    "raw_doviz_kurlari": df_dolar,
    "raw_hisse_fiyatlari": df_hisseler
}

print("🚀 Veriler BigQuery'ye aktarılıyor...")
for tablo_adi, df_tablo in tablolar.items():
    hedef_tablo = f"{PROJECT_ID}.{DATASET_ID}.{tablo_adi}"
    job = client.load_table_from_dataframe(df_tablo, hedef_tablo, job_config=job_config)
    job.result()  # Yüklemenin tamamlanmasını bekle
    print(f"✅ '{tablo_adi}' tablosu BigQuery'ye yüklendi! ({len(df_tablo)} satır)")

print("\n🎉 Tüm tablolar bulut ambarında hazır!")

🚀 Veriler BigQuery'ye aktarılıyor...
✅ 'dim_sirketler' tablosu BigQuery'ye yüklendi! (5 satır)
✅ 'raw_doviz_kurlari' tablosu BigQuery'ye yüklendi! (66 satır)
✅ 'raw_hisse_fiyatlari' tablosu BigQuery'ye yüklendi! (330 satır)

🎉 Tüm tablolar bulut ambarında hazır!
